# Path-Dependent Volatility

Source-style PDV checkpoint diagnostics with released/local checkpoint modes.


## Setup and plotting style


In [ ]:
import os
import shlex
from pathlib import Path

MODE = "released"  # choices: "released", "local"
SEED = 99
N_SAMPLE_TEST = 5000
RUN_SOURCE_DIAGNOSTICS = False
RUN_SIGNATURE_DIAGNOSTIC = False
RUN_FINANCE_DIAGNOSTICS = False
RUN_AWD_DIAGNOSTIC = False

os.environ.setdefault("MPLCONFIGDIR", "/tmp/time-causal-vae-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from time_causal_vae.evaluation.style import apply_source_style  # noqa: E402

apply_source_style()


def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root from a notebook or shell directory."""
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the time-causal-vae repository root.")


def resolve_path(path: str | Path) -> Path:
    """Resolve a path relative to the repository root."""
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    """Render a path relative to the repository root when possible."""
    candidate = Path(path).resolve()
    try:
        return str(candidate.relative_to(REPO_ROOT))
    except ValueError:
        return str(candidate)


def latest_local_final_model(output_root: Path, pattern: str) -> Path | None:
    """Return the newest local final_model directory matching a pattern."""
    candidates = sorted(output_root.glob(pattern), key=lambda candidate: candidate.stat().st_mtime)
    return candidates[-1] if candidates else None


def squeeze_paths(tensor: torch.Tensor) -> torch.Tensor:
    """Convert path tensors to two-dimensional CPU tensors for plotting."""
    values = tensor.detach().cpu().float()
    if values.ndim == 3 and values.shape[-1] == 1:
        values = values[..., 0]
    return values


REPO_ROOT = find_repo_root()
CONFIG_PATH = resolve_path("configs/experiments/pdv_info_cvae.yaml")
RELEASED_MODEL_DIR = resolve_path(
    "../TimeCausalVAE/trained_models/PDVPriceConFeature_timestep_60/model_InfoCVAE_De_CLSTMRes_En_CLSTMRes_Prior_RealNVP_Con_Id_Dis_None_comment_None/InfoCVAE_training_2024-08-21_16-06-50/final_model"
)
LOCAL_OUTPUT_ROOT = resolve_path("outputs/full/pdv_info_cvae")
LOCAL_PATTERN = "InfoCVAE_training_*/final_model"
BASE_DATA_DIR = resolve_path("data/processed")

if MODE not in {"released", "local"}:
    raise ValueError("MODE must be either 'released' or 'local'.")

print(f"Repository root: {REPO_ROOT}")
print(f"Mode: {MODE}")
print(f"Seed: {SEED}")
print(f"n_sample_test: {N_SAMPLE_TEST}")
print("Heavy source diagnostics are opt-in; SAWD/AWD can take many minutes on CPU.")

## Configuration and checkpoint selection


In [ ]:
if MODE == "released":
    model_dir = RELEASED_MODEL_DIR
    evaluation_dir = resolve_path("outputs/released_target_eval_pdv_5000")
    experiment_dir = evaluation_dir
else:
    model_dir = latest_local_final_model(LOCAL_OUTPUT_ROOT, LOCAL_PATTERN)
    if model_dir is None:
        model_dir = LOCAL_OUTPUT_ROOT / "<training-run>" / "final_model"
    evaluation_dir = LOCAL_OUTPUT_ROOT / "evaluation"
    experiment_dir = LOCAL_OUTPUT_ROOT

evaluation_dir.mkdir(parents=True, exist_ok=True)
summary_path = evaluation_dir / "summary.json"
batch_path = evaluation_dir / "evaluation_batch.pt"

print(f"config: {display_path(CONFIG_PATH)}")
print(f"model_dir: {display_path(model_dir)}")
print(f"evaluation_dir: {display_path(evaluation_dir)}")
print(f"model_dir exists: {model_dir.exists()}")

selected_config = yaml.safe_load(CONFIG_PATH.read_text())
rows = [
    ("dataset", selected_config.get("dataset", {}).get("name")),
    ("objective", selected_config.get("model", {}).get("objective")),
    ("encoder", selected_config.get("model", {}).get("encoder")),
    ("decoder", selected_config.get("model", {}).get("decoder")),
    ("conditioner", selected_config.get("model", {}).get("conditioner")),
    ("prior", selected_config.get("model", {}).get("prior")),
    ("alpha", selected_config.get("model", {}).get("alpha")),
    ("beta", selected_config.get("model", {}).get("beta")),
    ("epochs", selected_config.get("training", {}).get("epochs")),
    ("n_sample", selected_config.get("dataset", {}).get("n_sample")),
    ("n_timestep", selected_config.get("dataset", {}).get("n_timestep")),
]
display(pd.DataFrame(rows, columns=["field", "value"]))

command = [
    "poetry",
    "run",
    "tcvae-evaluate",
    "--config",
    display_path(CONFIG_PATH),
    "--model-dir",
    display_path(model_dir),
    "--output-dir",
    display_path(evaluation_dir),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--n-sample-test",
    str(N_SAMPLE_TEST),
    "--seed",
    str(SEED),
]
print("Equivalent evaluation command:")
print(" ".join(shlex.quote(part) for part in command))

## Load model and generate real/fake/reconstruction data


In [ ]:
from time_causal_vae.evaluation.checkpoints import TargetModelEvaluator

if not model_dir.exists():
    evaluator = None
    real_data = fake_data = recon_data = None
    batch = {}
    print("Checkpoint not found. Switch MODE or train/evaluate a local checkpoint first.")
else:
    evaluator = TargetModelEvaluator(str(model_dir), base_data_dir=str(BASE_DATA_DIR))
    real_data, fake_data, recon_data = evaluator.load_data(n_sample_test=N_SAMPLE_TEST, seed=SEED)
    base_dataset = evaluator.ensure_base_dataset()
    batch = {
        "real_data": real_data.detach().cpu(),
        "fake_data": fake_data.detach().cpu(),
        "recon_data": recon_data.detach().cpu(),
    }
    display(
        pd.DataFrame(
            [
                {
                    "tensor": name,
                    "shape": tuple(tensor.shape),
                    "mean": float(tensor.float().mean()),
                    "std": float(tensor.float().std()),
                }
                for name, tensor in batch.items()
            ]
        )
    )

## Path comparison


In [ ]:
if not batch:
    print("Path comparison skipped because no checkpoint was loaded.")
else:
    apply_source_style()
    fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharex=True, sharey=True)
    for axis, (key, title) in zip(
        axes,
        [("real_data", "Real"), ("fake_data", "Generated"), ("recon_data", "Reconstruction")],
        strict=True,
    ):
        paths = squeeze_paths(batch[key])[:24]
        axis.plot(paths.T, alpha=0.45, linewidth=0.9)
        axis.set_title(title)
        axis.set_xlabel("Time")
    axes[0].set_ylabel("Path value")
    fig.tight_layout()

## Condition path


In [ ]:
if evaluator is None:
    print("Condition-path diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.conditional import plot_path_condition

    plot_path_condition(base_dataset.path, base_dataset.sigma[0, :, 0])

## Conditional generations


In [ ]:
if not RUN_SOURCE_DIAGNOSTICS:
    print("Conditional generation plot skipped. Set RUN_SOURCE_DIAGNOSTICS=True to run it.")
elif evaluator is None:
    print("Conditional generation plot skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.conditional import con_gen_plot

    conditions = np.linspace(0, 0.5, 8)
    con_gen_plot(
        evaluator.model, conditions, file_path=evaluation_dir / "conditional_generations.png"
    )

## Path extension


In [ ]:
if not RUN_SOURCE_DIAGNOSTICS:
    print("Path extension skipped. Set RUN_SOURCE_DIAGNOSTICS=True to run it.")
elif evaluator is None:
    print("Path extension skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.conditional import plot_path_extension

    extended_paths = plot_path_extension(
        base_dataset.path,
        evaluator.model,
        evaluator.exp_config,
        n_extend_path=5,
        n_extend_time=10,
        file_path=evaluation_dir / "path_extension.png",
    )
    print(f"extended paths: {len(extended_paths)}")

## Weak metrics: MMD and SWD


In [ ]:
if evaluator is None:
    print("Weak metrics skipped because no checkpoint was loaded.")
else:
    hyper_metric = evaluator.compute_hyper_metric(real_data, fake_data)
    display(
        pd.DataFrame([{key: float(value.detach().cpu()) for key, value in hyper_metric.items()}])
    )
    if RUN_SIGNATURE_DIAGNOSTIC:
        from time_causal_vae.evaluation.metrics import SignatureMMD

        try:
            signature_mmd = SignatureMMD()(real_data, fake_data)
            print(f"Signature MMD: {float(signature_mmd.detach().cpu())}")
        except ModuleNotFoundError as exc:
            print(f"Expected-signature diagnostic skipped: {exc}")
    else:
        print("Expected-signature diagnostic skipped by default.")

## Optional conditional metrics and SAWD


In [ ]:
if not RUN_AWD_DIAGNOSTIC:
    print("Conditional SAWD/AWD diagnostic skipped. It can take many minutes on CPU.")
elif evaluator is None:
    print("Conditional SAWD/AWD diagnostic skipped because no checkpoint was loaded.")
else:
    from time_causal_vae.evaluation.conditional import (
        compute_eval_awd_dist_con,
        load_data_eval_dist_con,
        plot_eval_awd_dist_con,
    )

    con_paths = load_data_eval_dist_con(
        5000, base_dataset, evaluator.model, evaluation_dir, plot=True
    )
    metric = compute_eval_awd_dist_con(con_paths, evaluation_dir)
    plot_eval_awd_dist_con(metric, file_path=evaluation_dir / "conditional_sawd.png")